# Train Spam Classifier on Google Colab

Use this notebook on a GPU runtime. It prepares the project, installs a Colab-safe dependency set, trains the DeBERTa contextual spam classifier, evaluates it, and exports artifacts to Google Drive.

## 1. Runtime

Before running cells, choose `Runtime > Change runtime type > T4 GPU` or better.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Load Project

Option A: set `REPO_URL` to clone from GitHub. Option B: leave it empty and upload the clean ZIP package.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

from google.colab import files

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

PROJECT_DIR = Path('/content/BTL-TTNT')
REPO_URL = ''  # Fill this if you have a GitHub repo; otherwise upload ZIP below.

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if REPO_URL.strip():
    !git clone "$REPO_URL" "$PROJECT_DIR"
else:
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
    if zip_name is None:
        raise RuntimeError('Upload BTL-TTNT-colab-clean.zip from the project root')

    extract_dir = Path('/content/_btl_extract')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    extract_dir_text = str(extract_dir)
    !unzip -q -o "{zip_name}" -d "{extract_dir_text}"

    matches = list(extract_dir.rglob('src/config.py'))
    if not matches:
        raise RuntimeError('ZIP is invalid: missing src/config.py')
    source_root = matches[0].parents[1]
    shutil.copytree(source_root, PROJECT_DIR)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('Project ready:', PROJECT_DIR)
!pwd
!ls -la
from src.config import TrainingConfig
print('Project import OK')

## 3. Install Dependencies

The project does not use `torchvision` or `torchaudio`. Removing them avoids Colab CUDA mismatch errors while keeping the preinstalled GPU PyTorch build.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/BTL-TTNT')
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

!pip uninstall -y torchvision torchaudio
!python -m pip install --upgrade pip
!pip install -r requirements.txt

from src.config import TrainingConfig
print('Dependency install complete')

## 4. Kaggle Credentials (Optional)

Run this only if Kaggle download asks for authentication. Preferred: add a Colab Secret named `KAGGLE_API_TOKEN` with the Kaggle token value. Legacy fallback: upload `kaggle.json`.

In [ ]:
import os
from google.colab import files
from pathlib import Path

kaggle_dir = Path('/root/.kaggle')
legacy_dir = Path('/root/.config/kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
legacy_dir.mkdir(parents=True, exist_ok=True)

token = os.environ.get('KAGGLE_API_TOKEN', '').strip()
try:
    from google.colab import userdata
    token = token or (userdata.get('KAGGLE_API_TOKEN') or '').strip()
except Exception:
    pass

if token:
    access_token = kaggle_dir / 'access_token'
    access_token.write_text(token)
    access_token.chmod(0o600)
    os.environ['KAGGLE_API_TOKEN'] = token
    print('Kaggle API token saved from KAGGLE_API_TOKEN secret')
else:
    print('No KAGGLE_API_TOKEN secret found. Upload legacy kaggle.json, or cancel to try public access.')
    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        for target in (kaggle_dir / 'kaggle.json', legacy_dir / 'kaggle.json'):
            target.write_bytes(uploaded['kaggle.json'])
            target.chmod(0o600)
        print('Legacy kaggle.json saved')
    elif 'access_token' in uploaded:
        target = kaggle_dir / 'access_token'
        target.write_bytes(uploaded['access_token'])
        target.chmod(0o600)
        print('Kaggle access_token file saved')
    else:
        print('No Kaggle credentials saved; public access will be tried')

## 5. Environment Check

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/BTL-TTNT')
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import torch
print('Project dir:', PROJECT_DIR)
print('Python:', sys.executable)
print('Torch:', torch.__version__)
print('Torch CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU detected. Use Runtime > Change runtime type > T4 GPU.')

from transformers import AutoConfig, AutoTokenizer
from src.config import TrainingConfig
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
config = AutoConfig.from_pretrained('microsoft/deberta-v3-base', num_labels=2)
print('Transformers smoke OK:', type(tokenizer).__name__, config.model_type, config.num_labels)
print('Project smoke OK')

## 6. Train DeBERTa Contextual Model

Expected time on T4 GPU: roughly 1-3 hours depending on Colab load and downloads.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/BTL-TTNT')
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.config import TrainingConfig
from src import config as config_module
from src.data import dataset as dataset_module
from src.models import train as train_module

colab_config = TrainingConfig(
    spamassassin_ham_sources=('easy_ham', 'easy_ham_2', 'hard_ham'),
    contextual_model_name='microsoft/deberta-v3-base',
    contextual_max_length=256,
    contextual_batch_size=8,
    contextual_eval_batch_size=16,
    contextual_epochs=3,
    contextual_learning_rate=1e-5,
    contextual_weight_decay=0.01,
    contextual_freeze_base_model=False,
    contextual_warmup_ratio=0.10,
    contextual_gradient_accumulation_steps=2,
    contextual_early_stopping_patience=1,
    contextual_use_class_weights=True,
)

config_module.DEFAULT_TRAINING_CONFIG = colab_config
dataset_module.DEFAULT_TRAINING_CONFIG = colab_config
train_module.DEFAULT_TRAINING_CONFIG = colab_config

print('Starting training')
print('Backbone:', colab_config.contextual_model_name)
print('Batch size:', colab_config.contextual_batch_size)
print('Epochs:', colab_config.contextual_epochs)
print('Ham augmentation:', colab_config.spamassassin_ham_sources)

metadata = train_module.train_and_save(mode='contextual')

print('TRAINING COMPLETE')
print('Best model:', metadata['best_model'])
print('Backbone:', metadata['contextual_backbone'])
print('Test metrics:')
for key, value in metadata['test_metrics'].items():
    if isinstance(value, float):
        print(f'  {key}: {value:.4f}')
    elif key not in {'confusion_matrix', 'classification_report'}:
        print(f'  {key}: {value}')

## 7. Evaluate

In [ ]:
!python -m src.evaluation.evaluate
!python -m src.evaluation.non_spam

## 8. Export Artifacts To Google Drive

In [ ]:
import shutil
import time
from pathlib import Path

artifact_root = Path('/content/drive/MyDrive/BTL-TTNT-artifacts')
artifact_root.mkdir(parents=True, exist_ok=True)
run_dir = artifact_root / f'colab-run-{int(time.time())}'
run_dir.mkdir(parents=True, exist_ok=True)

for folder_name in ('models', 'docs', 'data/processed'):
    source = Path(folder_name)
    if not source.exists():
        print('Missing, skipped:', source)
        continue
    target = run_dir / source
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(source, target)
    print('Exported:', source, '->', target)

print('Artifacts saved to:', run_dir)